In [ ]:
import cftime
import numpy as np
import xarray as xr
from datetime import datetime
from netCDF4 import Dataset, date2num

# 打开原始的NetCDF数据集
# ds = xr.open_dataset("D:/csj/CMFD/Temp/2009/temp_ITPCAS-CMFD_V0106_B-01_03hr_010deg_200911.nc")
# ds1 = xr.open_dataset("D:/csj/CMFD/Pres/2009/pres_ITPCAS-CMFD_V0106_B-01_03hr_010deg_200911.nc")
# ds2 = xr.open_dataset("D:/csj/CMFD/SHum/2009/shum_ITPCAS-CMFD_V0106_B-01_03hr_010deg_200911.nc")
# ds3 = xr.open_dataset("D:/csj/CMFD/Wind/2009/wind_ITPCAS-CMFD_V0106_B-01_03hr_010deg_200911.nc")
# ds4 = xr.open_dataset("D:/csj/CMFD/LRad/2009/lrad_ITPCAS-CMFD_V0106_B-01_03hr_010deg_200911.nc")
# ds5 = xr.open_dataset("D:/csj/CMFD/SRad/2009/srad_ITPCAS-CMFD_V0106_B-01_03hr_010deg_200911.nc")
# ds6 = xr.open_dataset("D:/csj/CMFD/Prec/2009/prec_ITPCAS-CMFD_V0106_B-01_03hr_010deg_200911.nc")

ds = xr.open_dataset("D:/csj/CMFD/Temp/temp_ITPCAS-CMFD_V0106_B-01_03hr_010deg_201006.nc")
ds1 = xr.open_dataset("D:/csj/CMFD/Pres/pres_ITPCAS-CMFD_V0106_B-01_03hr_010deg_201006.nc")
ds2 = xr.open_dataset("D:/csj/CMFD/SHum/shum_ITPCAS-CMFD_V0106_B-01_03hr_010deg_201006.nc")
ds3 = xr.open_dataset("D:/csj/CMFD/Wind/wind_ITPCAS-CMFD_V0106_B-01_03hr_010deg_201006.nc")
ds4 = xr.open_dataset("D:/csj/CMFD/LRad/lrad_ITPCAS-CMFD_V0106_B-01_03hr_010deg_201006.nc")
ds5 = xr.open_dataset("D:/csj/CMFD/SRad/srad_ITPCAS-CMFD_V0106_B-01_03hr_010deg_201006.nc")
ds6 = xr.open_dataset("D:/csj/CMFD/Prec/prec_ITPCAS-CMFD_V0106_B-01_03hr_010deg_201006.nc")

# 处理时间数据，将2009改成2010
convert_time = [
    cftime.DatetimeNoLeap(
        i.year, i.month, i.day, i.hour, i.minute, i.second, i.microsecond // 1000
    ) for i in [datetime.strptime(str(t), '%Y-%m-%dT%H:%M:%S.%f000') for t in ds['time'].values]
]

# 读取变量
time = ds['time'].values
lat = ds['lat'].values
lon = ds['lon'].values
temp = ds['temp'].values
pres = ds1['pres'].values
shum = ds2['shum'].values
wind = ds3['wind'].values
lrad = ds4['lrad'].values
srad = ds5['srad'].values
prec = ds6['prec'].values * 0.000278

# 选择经纬度范围（96°E-112°E，21°N-35°N）
lat_min, lat_max = 21, 35
lon_min, lon_max = 96, 112
lat_idx = np.where((lat >= lat_min) & (lat <= lat_max))[0]
lon_idx = np.where((lon >= lon_min) & (lon <= lon_max))[0]

# 裁剪数据
lat_clipped = lat[lat_idx]
lon_clipped = lon[lon_idx]
lrad_clipped = lrad[:, lat_idx, :][:, :, lon_idx]
temp_clipped = temp[:, lat_idx, :][:, :, lon_idx]
pres_clipped = pres[:, lat_idx, :][:, :, lon_idx]
shum_clipped = shum[:, lat_idx, :][:, :, lon_idx]
wind_clipped = wind[:, lat_idx, :][:, :, lon_idx]
srad_clipped = srad[:, lat_idx, :][:, :, lon_idx]
prec_clipped = prec[:, lat_idx, :][:, :, lon_idx]

# 创建 NetCDF 文件
dataset = Dataset('D:/csj/CMFD/ceshi1/CMFD201006_001001.nc', 'w', format='NETCDF4')

# 创建维度
dataset.createDimension('time', len(time))
dataset.createDimension('lat', len(lat_clipped))
dataset.createDimension('lon', len(lon_clipped))
dataset.createDimension('scalar', 1)

# 创建变量
time_var = dataset.createVariable('time', 'f4', ('time',))
LONGXY_var = dataset.createVariable('LONGXY', np.float32, ('lat', 'lon'))
LATIXY_var = dataset.createVariable('LATIXY', np.float32, ('lat', 'lon'))
EDGEE_var = dataset.createVariable('EDGEE', np.float32, ('scalar',))
EDGEW_var = dataset.createVariable('EDGEW', np.float32, ('scalar',))
EDGES_var = dataset.createVariable('EDGES', np.float32, ('scalar',))
EDGEN_var = dataset.createVariable('EDGEN', np.float32, ('scalar',))
FLDS_var = dataset.createVariable('FLDS', np.float32, ('time', 'lat', 'lon'), fill_value=0.0001)
QBOT_var = dataset.createVariable('QBOT', np.float32, ('time', 'lat', 'lon'), fill_value=0.0001)
TBOT_var = dataset.createVariable('TBOT', np.float32, ('time', 'lat', 'lon'), fill_value=273.15)
WIND_var = dataset.createVariable('WIND', np.float32, ('time', 'lat', 'lon'), fill_value=0.0001)
PSRF_var = dataset.createVariable('PSRF', np.float32, ('time', 'lat', 'lon'), fill_value=85000)
FSDS_var = dataset.createVariable('FSDS', np.float32, ('time', 'lat', 'lon'), fill_value=0.0001)
PRECTmms_var = dataset.createVariable('PRECTmms', np.float32, ('time', 'lat', 'lon'), fill_value=0.0001)

# 添加数据
time_var.long_name = "observation time"
time_var.units = "days since 2010-06-01 00:00:00"
time_var[:] = date2num(convert_time, units=time_var.units, calendar='noleap')
time_var.calendar = "noleap"
LONGXY_var[:] = np.meshgrid(lon_clipped, lat_clipped)[0]
LATIXY_var[:] = np.meshgrid(lon_clipped, lat_clipped)[1]
FLDS_var[:] = lrad_clipped
QBOT_var[:] = shum_clipped
TBOT_var[:] = temp_clipped
WIND_var[:] = wind_clipped
PSRF_var[:] = pres_clipped
FSDS_var[:] = srad_clipped
PRECTmms_var[:] = prec_clipped
# 添加属性
LONGXY_var.long_name = "longitude"
LONGXY_var.units = "degrees_east"
LONGXY_var.mode = "time-invariant"
LATIXY_var.long_name = "latitude"
LATIXY_var.units = "degrees_north"
LATIXY_var.mode = "time-invariant"
EDGEE_var.long_name = "eastern edge in atmospheric data"
EDGEE_var.units = "degrees_east"
EDGEE_var.mode = "time-invariant"
EDGEW_var.long_name = "western edge in atmospheric data"
EDGEW_var.units = "degrees_east"
EDGEW_var.mode = "time-invariant"
EDGES_var.long_name = "southern edge in atmospheric data"
EDGES_var.units = "degrees_north"
EDGES_var.mode = "time-invariant"
EDGEN_var.long_name = "northern edge in atmospheric data"
EDGEN_var.units = "degrees_north"
EDGEN_var.mode = "time-invariant"

PSRF_var.long_name = "surface pressure at the lowest atm level"
PSRF_var.units = "Pa"
PSRF_var.mode = "time-dependent"
PSRF_var.missing_value = np.float32(85000)

TBOT_var.long_name = "temperature at the lowest atm level"
TBOT_var.units = "K"
TBOT_var.mode = "time-dependent"
TBOT_var.missing_value = np.float32(273.15)

WIND_var.long_name = "wind at the lowest atm level"
WIND_var.units = "m/s"
WIND_var.mode = "time-dependent"
WIND_var.missing_value = np.float32(0.0001)

QBOT_var.long_name = "specific humidity at the lowest atm level"
QBOT_var.units = "kg/kg"
QBOT_var.mode = "time-dependent"
QBOT_var.missing_value = np.float32(0.0001)

FLDS_var.long_name = "incident longwave radiation"
FLDS_var.units = "W/m**2"
FLDS_var.mode = "time-dependent"
FLDS_var.missing_value = np.float32(0.0001)

FSDS_var.long_name = "total incident solar radiation"
FSDS_var.units = "W/m**2"
FSDS_var.mode = "time-dependent"
FSDS_var.missing_value = np.float32(0.0001)

PRECTmms_var.long_name = "PRECTmms total precipitation"
PRECTmms_var.units = "mm H2O / sec"
PRECTmms_var.mode = "time-dependent"
PRECTmms_var.missing_value = np.float32(0.0001)

# 关闭文件
dataset.close()


In [ ]:
import numpy as np
from netCDF4 import Dataset

# 打开 NetCDF 文件
file_path = 'D:/csj/CMFD/ceshi1/CMFD201006_001001.nc'
dataset = Dataset(file_path, 'r+')

# 打印文件中的所有变量和维度
print("Variables and dimensions in the file:")
print(dataset.variables)

# 设置替换值字典
replace_values = {
    'FLDS': np.float32(0.0001),
    'QBOT': np.float32(0.0001),
    'WIND': np.float32(0.0001),
    'FSDS': np.float32(0.0001),
    'PRECTmms': np.float32(0.0001),
    'TBOT': np.float32(273.15),
    'PSRF': np.float32(85000)
}

# 检查文件中的每个变量是否有 NaN 或 missing_value，并替换
for var_name in dataset.variables:
    var = dataset.variables[var_name]
    print(f"\nChecking and replacing values in variable: {var_name}")
    
    # 获取该变量的数据
    data = var[:]
    
    # 获取替换值，如果没有为该变量设置替换值，则跳过
    replace_value = replace_values.get(var_name, None)
    if replace_value is None:
        print(f"  No specific replacement value for {var_name}, skipping.")
        continue
    
    # 检查 NaN 并替换为 replace_value
    if np.isnan(data).any():
        print(f"  NaN found in {var_name}, replacing with {replace_value}")
        data[np.isnan(data)] = replace_value
    
    # 检查 missing_value 属性并替换
    missing_value = getattr(var, 'missing_value', None)
    if missing_value is not None:
        print(f"  Missing value found in {var_name} (value: {missing_value}), replacing with {replace_value}")
        data[data == missing_value] = replace_value

    # 特别处理 'PRECTmms' 变量，替换负值为 replace_value，并且除以3600
    if var_name == 'PRECTmms':
        print(f"  Checking negative values in {var_name}, replacing with {replace_value}")
        # 先除以 3600
        data = data / 3600.0
        
        # 替换负值为指定的值
        data[data < 0] = replace_value
    
    # 将替换后的数据写回 NetCDF 文件
    var[:] = data

# 关闭文件
dataset.close()

print("Replacement complete. All NaN, missing values, and negative values in PRECTmms have been replaced with the specified values.")


In [4]:
import xarray as xr
ds= xr.open_dataset('D:/csj/CMFD/CMFD200901_001001.nc')
ds


<xarray.Dataset>
Dimensions:   (time: 248, lat: 140, lon: 160, scalar: 1)
Coordinates:
  * time      (time) object 2008-01-01 00:00:00 ... 2008-01-31 21:00:00
Dimensions without coordinates: lat, lon, scalar
Data variables: (12/13)
    LONGXY    (lat, lon) float32 ...
    LATIXY    (lat, lon) float32 ...
    EDGEE     (scalar) float32 ...
    EDGEW     (scalar) float32 ...
    EDGES     (scalar) float32 ...
    EDGEN     (scalar) float32 ...
    ...        ...
    QBOT      (time, lat, lon) float32 ...
    TBOT      (time, lat, lon) float32 ...
    WIND      (time, lat, lon) float32 ...
    PSRF      (time, lat, lon) float32 ...
    FSDS      (time, lat, lon) float32 ...
    PRECTmms  (time, lat, lon) float32 ...

In [7]:
import xarray as xr
ds= xr.open_dataset('D:/csj/CMFD/CMFD200901_001001.nc')
ds


<xarray.Dataset>
Dimensions:   (time: 248, lat: 140, lon: 160, scalar: 1)
Coordinates:
  * time      (time) object 2008-01-01 00:00:00 ... 2008-01-31 21:00:00
Dimensions without coordinates: lat, lon, scalar
Data variables: (12/13)
    LONGXY    (lat, lon) float32 ...
    LATIXY    (lat, lon) float32 ...
    EDGEE     (scalar) float32 ...
    EDGEW     (scalar) float32 ...
    EDGES     (scalar) float32 ...
    EDGEN     (scalar) float32 ...
    ...        ...
    QBOT      (time, lat, lon) float32 ...
    TBOT      (time, lat, lon) float32 ...
    WIND      (time, lat, lon) float32 ...
    PSRF      (time, lat, lon) float32 ...
    FSDS      (time, lat, lon) float32 ...
    PRECTmms  (time, lat, lon) float32 ...